# From Tokens to Circuits — Parte 1: de los tokens a un GPT

**Taller: An Introduction to Transformers and Mechanistic Interpretability**

En este notebook construimos, pieza a pieza, un modelo tipo GPT y lo entrenamos con las obras de Shakespeare. Sigue el recorrido de los capítulos 2–4 de *Build a Large Language Model (From Scratch)* de Sebastian Raschka:

1. **Texto → tokens**: tokenización BPE y cómo se preparan los pares entrada/objetivo.
2. **Tokens → vectores**: embeddings de token y de posición.
3. **Atención**: de la versión más simple a *multi-head attention* causal.
4. **El GPT completo**: LayerNorm, GELU, bloque transformer, modelo.
5. **Entrenamiento rápido** y generación de texto.
6. **Un primer vistazo dentro**: los patrones de atención del modelo que acabamos de entrenar.

> 💡 En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. También funciona en CPU, solo que entrena menos pasos.

Las celdas marcadas con 🧪 son para jugar: cambia valores y vuelve a ejecutar. Las celdas `anim(...)` reproducen las animaciones de las diapositivas.

In [ ]:
!pip -q install tiktoken

In [ ]:
import math, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt

torch.manual_seed(123)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Dispositivo:", device)

In [ ]:
#@title 🎨 Estilo del taller (ejecuta esta celda; puedes ignorar su código)
import base64, html, io, logging, re, urllib.request, uuid
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display

# Paleta de las diapositivas
INK, MUTED, PAPER, PAPER2, LINE = "#13211A", "#45554C", "#F2F5EF", "#E1EADD", "#DAE4D5"
GREEN, GREEN_D, GREEN_L, BLUE, DARK = "#2FA36B", "#1B7347", "#6BD49C", "#3B82C4", "#0F1B16"

def _load_font(family, weights="400;600;700"):
    try:
        url = "https://fonts.googleapis.com/css2?family=" + family.replace(" ", "+") + ":wght@" + weights
        css = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Python"}), timeout=10).read().decode()
        for i, u in enumerate(re.findall(r"url\((https://[^)]+\.ttf)\)", css)):
            path = f"/tmp/{family.replace(' ', '_')}_{i}.ttf"
            open(path, "wb").write(urllib.request.urlopen(u, timeout=20).read())
            font_manager.fontManager.addfont(path)
        return True
    except Exception as e:
        print(f"(No se pudo descargar {family}; se usará la fuente por defecto: {e})")
        return False

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
_sans = _load_font("Google Sans")
_mono = _load_font("Google Sans Code", "400;600")
mpl.rcParams.update({
    "font.family": ["Google Sans", "DejaVu Sans"] if _sans else ["DejaVu Sans"],
    "font.monospace": ["Google Sans Code", "DejaVu Sans Mono"],
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": LINE, "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "grid.color": LINE, "grid.alpha": 1.0,
    "axes.prop_cycle": mpl.cycler(color=[GREEN, BLUE, INK, "#7D8C82", GREEN_L, "#8CBBE3"]),
    "lines.linewidth": 2.4, "lines.solid_capstyle": "round", "legend.frameon": False,
    "figure.dpi": 110, "image.cmap": "taller",
})
for _name, _cols in {
    "taller": [PAPER, "#B5E0C6", GREEN, GREEN_D, DARK],      # secuencial: crema → verde → casi negro
    "taller_div": [BLUE, "#B9D5EF", PAPER, "#9ED6B5", GREEN_D], # divergente: azul ← crema → verde
}.items():
    try: mpl.colormaps.register(LinearSegmentedColormap.from_list(_name, _cols), name=_name)
    except ValueError: pass

# Animaciones de las diapositivas, con el mismo código que en la presentación
def anim(name, max_width=980):
    """Muestra una animación de las diapositivas del taller."""
    w, h, body = ANIMS[name]
    doc = ('<!doctype html><html><head><meta charset="utf-8">'
           '<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Sans:wght@400..700'
           '&family=Google+Sans+Code:wght@400..600&display=swap"></head><body style="margin:0">' + body + '</body></html>')
    uid = "a" + uuid.uuid4().hex[:8]
    s0 = min(1, max_width / w)
    display(HTML(
        f'<div id="{uid}" style="width:100%;max-width:{max_width}px;height:{h*s0:.0f}px;position:relative;overflow:hidden;border-radius:14px">'
        f'<iframe srcdoc="{html.escape(doc, quote=True)}" scrolling="no" '
        f'style="position:absolute;left:0;top:0;width:{w}px;height:{h}px;border:0;transform:scale({s0});transform-origin:0 0"></iframe></div>'
        f'<script>(function(){{var d=document.getElementById("{uid}"),f=d.firstChild;'
        f'function fit(){{var s=Math.min(1,d.clientWidth/{w});f.style.transform="scale("+s+")";d.style.height=({h}*s)+"px"}}'
        f'fit();if(window.ResizeObserver)new ResizeObserver(fit).observe(d);}})();</script>'))

ANIMS = {}
ANIMS['nexttoken'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n#p{position:absolute;left:0;top:30px;width:1664px;display:flex;flex-wrap:wrap;gap:12px}\n.t{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:34px;padding:12px 18px;border-radius:12px;background:#DAE4D5;white-space:pre;transition:all .5s}\n.n{background:#2FA36B;color:#13211A}\n#b{position:absolute;left:0;top:190px;width:1100px}\n.r{display:flex;align-items:center;gap:20px;height:64px}\n.l{width:180px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;text-align:right;white-space:pre}\n.bar{height:40px;border-radius:8px;background:#3B82C4;transition:width .7s;width:0}\n.v{font-size:28px;color:#45554C;width:120px}\n#c{position:absolute;left:1180px;top:200px;width:480px;font-size:28px;line-height:1.45;color:#45554C}\n#c b{color:#13211A}\n</style>\n<div id="p"></div><div id="b"></div>\n<div id="c"><b>GPT-2 small, probabilidades reales.</b><br>El modelo puntúa los ~50.000 tokens del vocabulario, elegimos uno, lo añadimos y volvemos a empezar.</div>\n<script>\nconst S=[[[" be",0.984],[",",0.002],[" have",0.002],[" become",0.002],[" exist",0.0]],[[",",0.721],["?",0.065],[".",0.047],[":",0.03],["\\\\n",0.01]],[[" a",0.236],[" an",0.049],[" is",0.034],[" the",0.022],[" or",0.02]],[[" member",0.026],[" good",0.024],[" man",0.023],[" woman",0.018],[" person",0.017]]];\nconst base=["To"," be",","," or"," not"," to"];\nconst P=document.getElementById("p"),B=document.getElementById("b");\nlet k=0;\nfunction chip(t,c){const d=document.createElement("span");d.className="t"+(c?" n":"");d.textContent=t;P.appendChild(d);return d}\nfunction reset(){P.innerHTML="";base.forEach(t=>chip(t));k=0}\nfunction show(){B.innerHTML="";const top=S[k],mx=top[0][1];\n top.forEach((x,i)=>{const r=document.createElement("div");r.className="r";\n  r.innerHTML=\'<span class="l"></span><div class="bar"></div><span class="v"></span>\';\n  r.children[0].textContent=x[0];r.children[2].textContent=(x[1]*100).toFixed(1)+"%";\n  if(i==0)r.children[1].style.background="#2FA36B";B.appendChild(r);\n  setTimeout(()=>r.children[1].style.width=Math.max(6,x[1]/mx*760)+"px",60+i*90)});}\nfunction tick(){if(k>=S.length){reset();show();return}\n const t=S[k][0][0];setTimeout(()=>{const c=chip(t,1);c.style.opacity=0;requestAnimationFrame(()=>c.style.opacity=1);\n  setTimeout(()=>c.classList.remove("n"),1200);k++;if(k<S.length)setTimeout(show,700)},1600);}\nreset();show();setInterval(tick,3400);\n</script>')
ANIMS['tokenize'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n.lab{font-size:26px;color:#45554C;margin-bottom:14px}\n.row{display:flex;flex-wrap:wrap;gap:8px;min-height:84px}\n.t{display:flex;flex-direction:column;align-items:center;border-radius:10px;padding:8px 10px;transition:all .45s;opacity:0;transform:translateY(12px)}\n.t.on{opacity:1;transform:none}\n.s{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:32px;white-space:pre}\n.i{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:24px;color:#45554C;opacity:0;transition:opacity .4s;margin-top:4px}\n.t.id .i{opacity:1}\n#stage{position:absolute;left:0;top:10px;width:1664px}\n#count{font-size:30px;margin-top:22px;color:#13211A;opacity:0;transition:opacity .5s}\n</style>\n<div id="stage"><div class="lab" id="lab"></div><div class="row" id="row"></div><div id="count"></div></div>\n<script>\nconst C=["#B5E0C6","#B9D5EF","#9ED6B5","#A9CBEA","#C8EAD5","#C8DDF2"];\nconst X=[{lab:"Inglés · tokenizador BPE de GPT-2",t:["To"," be",","," or"," not"," to"," be",":"," that"," is"," the"," question","."],i:[2514,307,11,393,407,284,307,25,326,318,262,1808,13],n:42},\n{lab:"Castellano · el mismo tokenizador",t:["H","ola",","," b","ien","ven","id","os"," al"," taller"," de"," interpret","abil","idad"," me","can","ic","ista","."],i:[39,5708,11,275,2013,574,312,418,435,25242,390,6179,14991,32482,502,5171,291,12523,13],n:61}];\nconst R=document.getElementById("row"),L=document.getElementById("lab"),N=document.getElementById("count");let k=0;\nfunction run(){const x=X[k];R.innerHTML="";N.style.opacity=0;L.textContent=x.lab;\n const els=x.t.map((t,j)=>{const d=document.createElement("div");d.className="t";d.style.background=C[j%C.length];\n  d.innerHTML=\'<span class="s"></span><span class="i"></span>\';d.children[0].textContent=t.replace(/ /g,"␣");d.children[1].textContent=x.i[j];R.appendChild(d);return d});\n els.forEach((d,j)=>setTimeout(()=>d.classList.add("on"),300+j*170));\n const t1=300+els.length*170+500;\n els.forEach((d,j)=>setTimeout(()=>d.classList.add("id"),t1+j*80));\n setTimeout(()=>{N.textContent=x.n+" caracteres → "+x.t.length+" tokens";N.style.opacity=1},t1+els.length*80+300);\n k=(k+1)%X.length;}\nrun();setInterval(run,9000);\n</script>')
ANIMS['bpe'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n#w{position:absolute;left:0;top:20px;width:1000px}\n.r{display:flex;align-items:center;gap:8px;height:88px}\n.c{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:36px;padding:10px 16px;border-radius:10px;background:#DAE4D5;transition:all .4s}\n.h{background:#2FA36B}\n.m{background:#3B82C4;color:#F2F5EF}\n.k{font-size:28px;color:#45554C;margin-left:18px}\n#v{position:absolute;left:1080px;top:20px;width:580px}\n#v h{display:block;font-size:28px;color:#45554C;margin-bottom:16px}\n#list{display:flex;flex-wrap:wrap;gap:10px}\n.vv{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;padding:8px 14px;border-radius:10px;background:#0F1B16;color:#EAF2EA}\n#msg{position:absolute;left:0;top:400px;width:1000px;font-size:30px;color:#13211A}\n</style>\n<div id="w"></div><div id="v"><h>Tokens nuevos del vocabulario</h><div id="list"></div></div><div id="msg"></div>\n<script>\nconst C0=[["low",5],["lower",2],["newest",6],["widest",3]];\nlet W,step;const Wd=document.getElementById("w"),Ls=document.getElementById("list"),M=document.getElementById("msg");\nfunction init(){W=C0.map(([w,n])=>({s:w.split(""),n}));step=0;Ls.innerHTML="";draw();M.textContent="Empezamos con caracteres sueltos."}\nfunction draw(hp,mg){Wd.innerHTML="";W.forEach(w=>{const r=document.createElement("div");r.className="r";\n for(let i=0;i<w.s.length;i++){const c=document.createElement("span");c.className="c";c.textContent=w.s[i];\n  if(hp&&i<w.s.length-1&&w.s[i]==hp[0]&&w.s[i+1]==hp[1]){c.classList.add("h")}\n  if(hp&&i>0&&w.s[i-1]==hp[0]&&w.s[i]==hp[1])c.classList.add("h");\n  if(mg&&w.s[i]==mg)c.classList.add("m");r.appendChild(c)}\n const k=document.createElement("span");k.className="k";k.textContent="× "+w.n;r.appendChild(k);Wd.appendChild(r)})}\nfunction best(){const cnt={};W.forEach(w=>{for(let i=0;i<w.s.length-1;i++){const p=w.s[i]+"\\u0001"+w.s[i+1];cnt[p]=(cnt[p]||0)+w.n}});\n let b=null,bn=0;for(const p in cnt)if(cnt[p]>bn){b=p;bn=cnt[p]}return b?[b.split("\\u0001"),bn]:null}\nfunction merge(p){const t=p[0]+p[1];W.forEach(w=>{const o=[];for(let i=0;i<w.s.length;i++){if(i<w.s.length-1&&w.s[i]==p[0]&&w.s[i+1]==p[1]){o.push(t);i++}else o.push(w.s[i])}w.s=o});return t}\nfunction tick(){if(step>=7){init();return}\n const b=best();if(!b){init();return}const [p,n]=b;\n draw(p);M.textContent="Par más frecuente: «"+p[0]+"» + «"+p[1]+"» aparece "+n+" veces";\n setTimeout(()=>{const t=merge(p);draw(null,t);const v=document.createElement("span");v.className="vv";v.textContent=t;Ls.appendChild(v);M.textContent="→ nuevo token «"+t+"»"},1500);step++}\ninit();setInterval(tick,3000);\n</script>')
ANIMS['window'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n.strip{position:absolute;left:0;display:flex;gap:0}\n.t{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:22px;width:118px;height:64px;display:flex;align-items:center;justify-content:center;white-space:pre;border-radius:10px;transition:all .4s;color:#7D8C82}\n.x{background:#3B82C4;color:#F2F5EF}\n.y{background:#2FA36B;color:#13211A}\n.lab{position:absolute;left:0;font-size:26px;color:#45554C}\n#pairs{position:absolute;left:0;top:300px;width:1664px;display:flex;flex-direction:column;gap:10px}\n.p{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:28px;opacity:0;transition:opacity .4s;white-space:pre}\n.p b{color:#2B66A3;font-weight:600}.p i{font-style:normal;color:#1B7347;font-weight:600}\n</style>\n<div class="lab" style="top:0">entrada x</div><div class="strip" id="a" style="top:36px"></div>\n<div class="lab" style="top:130px">objetivo y</div><div class="strip" id="b" style="top:166px"></div>\n<div id="pairs"></div>\n<script>\nconst T=["First"," Citizen",":","⏎","Before"," we"," proceed"," any"," further",","," hear"," me"," speak",".","⏎","All",":"];\nconst n=4,A=document.getElementById("a"),B=document.getElementById("b"),P=document.getElementById("pairs");\nfunction mk(el){el.innerHTML="";return T.slice(0,14).map(t=>{const d=document.createElement("div");d.className="t";d.textContent=t;el.appendChild(d);return d})}\nconst ea=mk(A),eb=mk(B);let s=0;\nfunction tick(){ea.forEach(e=>e.className="t");eb.forEach(e=>e.className="t");P.innerHTML="";\n for(let i=s;i<s+n;i++){ea[i].classList.add("x");eb[i+1]&&eb[i+1].classList.add("y")}\n for(let j=1;j<=n;j++){const d=document.createElement("div");d.className="p";\n  d.innerHTML="<b></b>  →  <i></i>";d.children[0].textContent=T.slice(s,s+j).join("").replace("⏎","\\\\n");d.children[1].textContent=T[s+j].replace("⏎","\\\\n");\n  P.appendChild(d);setTimeout(()=>d.style.opacity=1,500+j*450)}\n s=s+n>=9?0:s+n}\ntick();setInterval(tick,4200);\n</script>')
ANIMS['whyembed'] = (1664, 620, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n.pan{position:absolute;top:0;height:600px;border-radius:24px;box-sizing:border-box;padding:36px}\nh3{margin:0;font-size:34px;font-weight:600}\n.q{font-size:26px;color:#45554C;margin-top:8px;line-height:1.35}\ntable{border-collapse:separate;border-spacing:4px;margin-top:34px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:24px}\ntd{width:46px;height:40px;text-align:center;border-radius:6px;background:#DAE4D5;color:#7D8C82}\ntd.w{width:130px;background:none;text-align:right;color:#13211A;padding-right:10px}\ntd.one{background:#3B82C4;color:#F2F5EF;font-weight:600}\n.dist{font-size:26px;color:#13211A;margin-top:28px;line-height:1.4}\n#pl{position:absolute;left:36px;top:150px;width:780px;height:420px}\n.pt{position:absolute;width:20px;height:20px;border-radius:10px;transition:left 2.2s cubic-bezier(.45,0,.25,1),top 2.2s cubic-bezier(.45,0,.25,1)}\n.lb{position:absolute;font-size:26px;white-space:nowrap;transition:left 2.2s cubic-bezier(.45,0,.25,1),top 2.2s cubic-bezier(.45,0,.25,1)}\n.cl{position:absolute;font-size:24px;font-weight:600;letter-spacing:1px;text-transform:uppercase;transition:opacity .8s}\n#st{position:absolute;right:36px;top:44px;font-size:26px;font-weight:600;color:#1B7347}\n</style>\n<div class="pan" style="left:0;width:760px;background:#E1EADD">\n<h3>Ids o one-hot</h3><div class="q">Cada palabra es un 1 en su columna de un vector de 50.257 ceros.</div>\n<table id="oh"></table>\n<div class="dist">«perro» está igual de lejos de «gato» que de «Madrid».<br>Ninguna noción de parecido, y vectores gigantes.</div></div>\n<div class="pan" style="left:800px;width:864px;background:#0F1B16;color:#EAF2EA">\n<h3>Embedding aprendido</h3><div class="q" style="color:#A2B8AB">Un vector denso, por ejemplo de 768 números, que se ajusta al entrenar.</div>\n<div id="st"></div><div id="pl"></div></div>\n<script>\nconst W=["perro","gato","Madrid","París","correr"];\nconst oh=document.getElementById("oh");W.forEach((w,i)=>{const r=document.createElement("tr");let h=\'<td class="w"></td>\';for(let j=0;j<8;j++)h+=j==7?\'<td>…</td>\':`<td class="${j==i?\'one\':\'\'}">${j==i?1:0}</td>`;r.innerHTML=h;r.children[0].textContent=w;oh.appendChild(r)});\nconst P=[["perro","animal",120,110],["gato","animal",200,60],["caballo","animal",90,200],["Madrid","ciudad",560,90],["París","ciudad",640,170],["Vigo","ciudad",520,200],["correr","verbo",300,330],["saltar","verbo",400,370],["nadar","verbo",230,390]];\nconst col={animal:"#6BD49C",ciudad:"#6FAEE8",verbo:"#EAF2EA"};\nconst pl=document.getElementById("pl");let seed=11;const rnd=()=>{seed=(seed*16807)%2147483647;return seed/2147483647};\nconst els=P.map(([w,c])=>{const d=document.createElement("div");d.className="pt";d.style.background=col[c];const l=document.createElement("div");l.className="lb";l.textContent=w;l.style.color=col[c];pl.appendChild(d);pl.appendChild(l);return [d,l]});\nconst C=[["animales",40,0,"#6BD49C"],["ciudades",520,10,"#6FAEE8"],["verbos",200,270,"#EAF2EA"]].map(([t,x,y,c])=>{const d=document.createElement("div");d.className="cl";d.textContent=t;d.style.left=x+"px";d.style.top=y+"px";d.style.color=c;d.style.opacity=0;pl.appendChild(d);return d});\nconst st=document.getElementById("st");\nfunction place(learned){els.forEach(([d,l],i)=>{const [,, x,y]=P[i],X=learned?x:30+rnd()*700,Y=learned?y:30+rnd()*360;d.style.left=X+"px";d.style.top=Y+"px";l.style.left=X+28+"px";l.style.top=Y-8+"px"});C.forEach(c=>c.style.opacity=learned?1:0);st.textContent=learned?"después de entrenar":"al inicio: al azar"}\nplace(true);\nfunction loop(){place(false);setTimeout(()=>place(true),2600)}\nsetTimeout(()=>{loop();setInterval(loop,8000)},3000);\n</script>')
ANIMS['embed'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n.lab{position:absolute;font-size:26px;color:#45554C}\n.mono{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace}\n.g{position:absolute;display:grid;gap:3px}\n.cell{width:30px;height:24px;border-radius:3px;transition:outline .3s}\n.vec{position:absolute;display:flex;gap:3px}\n.vec .cell{height:40px;width:42px;transition:background .5s}\n#tok{position:absolute;left:0;top:200px;width:200px;font-size:40px;text-align:center;background:#0F1B16;color:#EAF2EA;border-radius:14px;padding:16px 0}\n#tid{position:absolute;left:0;top:290px;width:200px;text-align:center;font-size:26px}\n.op{position:absolute;font-size:56px;color:#45554C}\n.hl{position:absolute;border:4px solid #2FA36B;border-radius:6px;transition:top .5s;pointer-events:none}\n</style>\n<div id="tok" class="mono"></div><div id="tid" class="mono"></div>\n<div class="lab" style="left:270px;top:0">tabla de tokens<br><span class="mono">[vocab × d]</span></div>\n<div class="g" id="E" style="left:270px;top:80px"></div><div class="hl" id="hE"></div>\n<div class="lab" style="left:640px;top:0">tabla de posiciones<br><span class="mono">[contexto × d]</span></div>\n<div class="g" id="P" style="left:640px;top:80px"></div><div class="hl" id="hP"></div>\n<div class="lab" style="left:1010px;top:60px">token</div><div class="vec" id="v1" style="left:1010px;top:100px"></div>\n<div class="op" style="left:1010px;top:150px">+</div>\n<div class="lab" style="left:1010px;top:230px">posición</div><div class="vec" id="v2" style="left:1010px;top:270px"></div>\n<div class="op" style="left:1010px;top:320px">=</div>\n<div class="lab" style="left:1010px;top:400px">entrada al transformer</div><div class="vec" id="v3" style="left:1010px;top:440px"></div>\n<script>\nconst D=8,R=14;let seed=7;const rnd=()=>{seed=(seed*16807)%2147483647;return seed/2147483647*2-1};\nconst col=v=>{const a=Math.max(-1,Math.min(1,v));return a>=0?`rgb(${242-a*195},${245-a*82},${239-a*132})`:`rgb(${242+a*183},${245+a*115},${239+a*43})`};\nconst Ev=[...Array(R)].map(()=>[...Array(D)].map(rnd)),Pv=[...Array(R)].map(()=>[...Array(D)].map(()=>rnd()*0.6));\nfunction grid(id,M){const g=document.getElementById(id);g.style.gridTemplateColumns=`repeat(${D},30px)`;M.forEach(r=>r.forEach(v=>{const c=document.createElement("div");c.className="cell";c.style.background=col(v);g.appendChild(c)}))}\ngrid("E",Ev);grid("P",Pv);\nfunction vec(id){const e=document.getElementById(id);for(let i=0;i<D;i++){const c=document.createElement("div");c.className="cell";c.style.background="#DAE4D5";e.appendChild(c)}return [...e.children]}\nconst v1=vec("v1"),v2=vec("v2"),v3=vec("v3");\nconst S=[["To",2514,3,0],["␣be",307,9,1],[",",11,5,2],["␣or",393,11,3]];let k=0;\nconst hE=document.getElementById("hE"),hP=document.getElementById("hP");\n[hE,hP].forEach((h,i)=>{h.style.left=(i?640:270)-4+"px";h.style.width=D*33+5+"px";h.style.height="27px"});\nfunction tick(){const [t,id,row,pos]=S[k];document.getElementById("tok").textContent=t;document.getElementById("tid").textContent="id "+id+" · pos "+pos;\n [v1,v2,v3].forEach(v=>v.forEach(c=>c.style.background="#DAE4D5"));\n hE.style.top=80+row*27-4+"px";hP.style.top=80+pos*27-4+"px";\n setTimeout(()=>v1.forEach((c,i)=>c.style.background=col(Ev[row][i])),700);\n setTimeout(()=>v2.forEach((c,i)=>c.style.background=col(Pv[pos][i])),1400);\n setTimeout(()=>v3.forEach((c,i)=>c.style.background=col(Ev[row][i]+Pv[pos][i])),2200);\n k=(k+1)%S.length}\ntick();setInterval(tick,3600);\n</script>')
ANIMS['simpleattn'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n.mono{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace}\n#words{position:absolute;left:0;top:380px;display:flex;gap:14px}\n.w{width:150px;height:64px;border-radius:12px;background:#DAE4D5;display:flex;align-items:center;justify-content:center;font-size:30px;transition:all .4s}\n.w.q{background:#0F1B16;color:#EAF2EA}\n#bars{position:absolute;left:0;top:60px;display:flex;gap:14px;height:300px;align-items:flex-end}\n.bw{width:150px;display:flex;flex-direction:column;align-items:center;justify-content:flex-end;height:300px}\n.bar{width:90px;background:#3B82C4;border-radius:8px 8px 0 0;height:0;transition:height .6s}\n.val{font-size:26px;margin-bottom:8px;opacity:0;transition:opacity .4s}\n#steps{position:absolute;left:0;top:470px;font-size:28px;color:#45554C;width:1040px}\n#hm{position:absolute;left:1140px;top:60px;display:grid;grid-template-columns:repeat(6,70px);gap:4px}\n.c{width:70px;height:56px;border-radius:6px;background:#DAE4D5;transition:background .5s;display:flex;align-items:center;justify-content:center;font-size:20px}\n#hl{position:absolute;left:1140px;top:20px;font-size:26px;color:#45554C}\n</style>\n<div id="bars"></div><div id="words"></div><div id="steps"></div>\n<div id="hl">Matriz de pesos de atención</div><div id="hm"></div>\n<script>\nconst W=["Tu","viaje","empieza","con","un","paso"];\nconst X=[[0.43,0.15,0.89],[0.55,0.87,0.66],[0.57,0.85,0.64],[0.22,0.58,0.33],[0.77,0.25,0.10],[0.05,0.80,0.55]];\nconst dot=(a,b)=>a.reduce((s,v,i)=>s+v*b[i],0);\nconst S=X.map(a=>X.map(b=>dot(a,b))),A=S.map(r=>{const e=r.map(Math.exp),z=e.reduce((a,b)=>a+b);return e.map(v=>v/z)});\nconst Wd=document.getElementById("words"),Bd=document.getElementById("bars"),St=document.getElementById("steps"),H=document.getElementById("hm");\nconst we=W.map(t=>{const d=document.createElement("div");d.className="w";d.textContent=t;Wd.appendChild(d);return d});\nconst be=W.map(()=>{const d=document.createElement("div");d.className="bw";d.innerHTML=\'<span class="val"></span><div class="bar"></div>\';Bd.appendChild(d);return d});\nconst hc=[];for(let i=0;i<36;i++){const c=document.createElement("div");c.className="c";H.appendChild(c);hc.push(c)}\nlet q=0;\nfunction tick(){if(q==0)hc.forEach(c=>c.style.background="#DAE4D5");\n we.forEach((w,i)=>w.className="w"+(i==q?" q":""));\n be.forEach((b,i)=>{b.children[1].style.height="0px";b.children[0].style.opacity=0;b.children[1].style.background="#AEBDB0"});\n St.textContent="1 · puntuación = producto escalar de «"+W[q]+"» con cada token";\n setTimeout(()=>be.forEach((b,i)=>{b.children[0].textContent=S[q][i].toFixed(2);b.children[0].style.opacity=1;b.children[1].style.height=S[q][i]*110+"px"}),300);\n setTimeout(()=>{St.textContent="2 · softmax: los pesos son positivos y suman 1";be.forEach((b,i)=>{b.children[0].textContent=A[q][i].toFixed(2);b.children[1].style.background="#3B82C4";b.children[1].style.height=A[q][i]*700+"px"})},1900);\n setTimeout(()=>{St.textContent="3 · nuevo vector de «"+W[q]+"» = Σ peso × vector";\n  for(let j=0;j<6;j++){const a=A[q][j],t=Math.min(1,(a-0.1)/0.16);hc[q*6+j].style.background=`rgb(${218-t*159},${228-t*98},${213-t*17})`;}},3400);\n q=(q+1)%6}\ntick();setInterval(tick,4800);\n</script>')
ANIMS['selfattn'] = (860, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n#g{position:absolute;left:150px;top:50px;display:grid;grid-template-columns:repeat(6,100px);gap:5px}\n.c{width:100px;height:66px;border-radius:8px;display:flex;align-items:center;justify-content:center;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:24px;border:4px solid transparent;transition:all .5s}\n.rl{position:absolute;left:0;width:136px;text-align:right;font-size:26px;height:66px;display:flex;align-items:center;justify-content:flex-end}\n.cl{position:absolute;top:10px;width:100px;text-align:center;font-size:24px}\n#cap{position:absolute;left:0;top:518px;width:860px;font-size:24px;font-weight:600;color:#1B7347}\n</style>\n<div id="g"></div><div id="cap"></div>\n<script>\nconst W=["Tu","viaje","empieza","con","un","paso"],X=[[0.43,0.15,0.89],[0.55,0.87,0.66],[0.57,0.85,0.64],[0.22,0.58,0.33],[0.77,0.25,0.10],[0.05,0.80,0.55]];\nconst A=X.map(a=>{const s=X.map(b=>a.reduce((t,v,i)=>t+v*b[i],0)).map(Math.exp),z=s.reduce((p,q)=>p+q);return s.map(v=>v/z)});\nconst G=document.getElementById("g"),cells=[];\nW.forEach((w,i)=>{const r=document.createElement("div");r.className="rl";r.style.top=50+i*71+"px";r.textContent=w;document.body.appendChild(r);\n const c=document.createElement("div");c.className="cl";c.style.left=150+i*105+"px";c.textContent=w;document.body.appendChild(c);\n for(let j=0;j<6;j++){const d=document.createElement("div");d.className="c";d.textContent=A[i][j].toFixed(2);G.appendChild(d);cells.push(d)}});\nconst bg=v=>{const t=Math.min(1,Math.max(0,(v-0.1)/0.16));return `rgb(${218-t*159},${228-t*98},${213-t*17})`};\nconst base=()=>cells.forEach((d,n)=>{const v=A[n/6|0][n%6];d.style.background=bg(v);d.style.color=v>0.2?"#F2F5EF":"#13211A";d.style.borderColor="transparent";d.style.opacity=1});\nconst cap=document.getElementById("cap");\nfunction diag(){base();cells.forEach((d,n)=>{const i=n/6|0,j=n%6;if(i==j)d.style.borderColor="#13211A";else d.style.opacity=.35});cap.textContent="Diagonal: cuánto se atiende cada token a sí mismo (15–24 %)"}\nfunction viaje(){base();cells.forEach((d,n)=>{const i=n/6|0,j=n%6,m=A[i].indexOf(Math.max(...A[i]));if(j==m)d.style.borderColor="#2FA36B";else d.style.opacity=.35});cap.textContent="Máximo de cada fila: casi siempre «viaje», el vector más grande"}\nfunction sym(){base();const pairs=[[1,4],[3,5],[0,2]];cells.forEach((d,n)=>{const i=n/6|0,j=n%6;if(pairs.some(([a,b])=>(i==a&&j==b)||(i==b&&j==a)))d.style.borderColor="#3B82C4";else d.style.opacity=.35});cap.textContent="Puntuación simétrica: x·y = y·x, y sin nada que aprender"}\nconst steps=[diag,viaje,sym];let k=0;\nbase();cap.textContent="Pesos de la atención simple, sin parámetros";\nsetInterval(()=>{steps[k]();k=(k+1)%steps.length},3200);\n</script>')
ANIMS['mask'] = (1664, 620, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n#g{position:absolute;left:170px;top:70px;display:grid;grid-template-columns:repeat(6,96px);gap:5px}\n.c{width:96px;height:80px;border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:24px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;transition:all .5s}\n.rl{position:absolute;left:0;width:150px;text-align:right;font-size:26px;height:80px;display:flex;align-items:center;justify-content:flex-end;transition:color .3s}\n.cl{position:absolute;top:20px;width:96px;text-align:center;font-size:24px;transition:color .3s}\n#txt{position:absolute;left:900px;top:80px;width:740px}\n.s{font-size:32px;line-height:1.35;padding:18px 24px;border-radius:14px;margin-bottom:18px;color:#7D8C82;transition:all .4s}\n.s.on{background:#0F1B16;color:#EAF2EA}\n</style>\n<div id="g"></div><div id="txt">\n<div class="s" id="s0">1 · Puntuaciones de todos con todos</div>\n<div class="s" id="s1">2 · Por encima de la diagonal: −∞</div>\n<div class="s" id="s2">3 · Softmax por filas: el futuro recibe peso 0</div>\n<div class="s" id="s3">4 · Cada token solo ve el pasado y a sí mismo</div></div>\n<script>\nconst W=["Tu","viaje","empieza","con","un","paso"],X=[[0.43,0.15,0.89],[0.55,0.87,0.66],[0.57,0.85,0.64],[0.22,0.58,0.33],[0.77,0.25,0.10],[0.05,0.80,0.55]];\nconst S=X.map(a=>X.map(b=>a.reduce((s,v,i)=>s+v*b[i],0)));\nconst M=S.map((r,i)=>{const e=r.map((v,j)=>j<=i?Math.exp(v):0),z=e.reduce((a,b)=>a+b);return e.map(v=>v/z)});\nconst G=document.getElementById("g"),cells=[],rl=[],cl=[];\nfor(let i=0;i<6;i++){const r=document.createElement("div");r.className="rl";r.style.top=70+i*85+"px";r.textContent=W[i];document.body.appendChild(r);rl.push(r);\n const c=document.createElement("div");c.className="cl";c.style.left=170+i*101+"px";c.textContent=W[i];document.body.appendChild(c);cl.push(c);\n for(let j=0;j<6;j++){const d=document.createElement("div");d.className="c";G.appendChild(d);cells.push(d)}}\nconst blue=t=>`rgb(${218-t*159},${228-t*98},${213-t*17})`;\nfunction st(k){for(let i=0;i<4;i++)document.getElementById("s"+i).className="s"+(i==k?" on":"")}\nfunction phase0(){st(0);cells.forEach((d,n)=>{const i=n/6|0,j=n%6;d.style.background=blue((S[i][j]-0.2)/1.4);d.style.color=S[i][j]>0.9?"#F2F5EF":"#13211A";d.textContent=S[i][j].toFixed(2);d.style.opacity=1});rl.forEach(r=>r.style.color="#13211A");cl.forEach(r=>r.style.color="#13211A")}\nfunction phase1(){st(1);cells.forEach((d,n)=>{const i=n/6|0,j=n%6;if(j>i){d.style.background="#3A3F4A";d.style.color="#EAF2EA";d.textContent="−∞"}})}\nfunction phase2(){st(2);cells.forEach((d,n)=>{const i=n/6|0,j=n%6;if(j>i){d.style.background="#E7EEE3";d.textContent="0";d.style.color="#AEBDB0"}else{d.style.background=blue(M[i][j]*1.6);d.style.color=M[i][j]>0.45?"#F2F5EF":"#13211A";d.textContent=M[i][j].toFixed(2)}})}\nlet row=0;function phase3(){st(3);row=0;const iv=setInterval(()=>{cells.forEach((d,n)=>{const i=n/6|0;d.style.opacity=i==row?1:0.25});rl.forEach((r,i)=>r.style.color=i==row?"#1B7347":"#AEBDB0");cl.forEach((c,j)=>c.style.color=j<=row?"#2B66A3":"#AEBDB0");row++;if(row>5)clearInterval(iv)},700)}\nfunction loop(){phase0();setTimeout(phase1,2200);setTimeout(phase2,4200);setTimeout(phase3,6400)}\nloop();setInterval(loop,11500);\n</script>')
ANIMS['multihead'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#0F1B16;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#EAF2EA}\n#q{position:absolute;left:0;top:0;width:1664px;font-size:30px;color:#A2B8AB}\n#q b{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;color:#6BD49C;font-weight:600}\n#sent{position:absolute;left:0;top:50px;width:1664px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:26px;color:#6A8074;white-space:pre}\n.h{position:absolute;top:110px;width:380px}\n.g{display:grid;grid-template-columns:repeat(15,22px);gap:2px}\n.c{width:22px;height:18px;border-radius:2px;background:#1A2C23;transition:background .35s}\n.n{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:26px;color:#6FAEE8;margin-top:16px}\n.r{font-size:26px;color:#EAF2EA;margin-top:4px}\n.a{font-size:26px;color:#A2B8AB;margin-top:10px;min-height:36px}\n.a b{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;color:#6BD49C;font-weight:600}\n</style>\n<div id="q"></div><div id="sent"></div>\n<script>\nconst T=["⟨inicio⟩","When"," Mary"," and"," John"," went"," to"," the"," store",","," John"," gave"," a"," drink"," to"];\nconst H=[["L4H11","token anterior",[[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,1,0,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,1,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,1,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,1,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,1,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,1,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,1,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,1,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,0,1,0]]],\n["L4H4","sumidero: primer token",[[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[.98,.02,0,0,0,0,0,0,0,0,0,0,0,0,0],[.96,0,.04,0,0,0,0,0,0,0,0,0,0,0,0],[.61,.02,.18,.19,0,0,0,0,0,0,0,0,0,0,0],[.85,0,.04,.08,.03,0,0,0,0,0,0,0,0,0,0],[.64,.15,.04,.07,.01,.09,0,0,0,0,0,0,0,0,0],[.74,.03,.01,.02,.01,.13,.05,0,0,0,0,0,0,0,0],[.76,.03,.03,.03,.01,.03,.07,.04,0,0,0,0,0,0,0],[.7,.01,.01,.02,0,.04,.08,.09,.05,0,0,0,0,0,0],[.63,.08,.03,.06,.01,.04,.03,.04,.05,.04,0,0,0,0,0],[.56,.01,.05,.15,.05,.02,.01,.02,0,.02,.12,0,0,0,0],[.72,.01,.03,.04,.01,.01,.01,.01,.01,.01,.05,.09,0,0,0],[.76,0,.01,.01,0,.01,.01,.01,.01,.01,.01,.08,.06,0,0],[.59,.01,.01,.02,0,.01,.01,.03,.01,.02,0,.12,.1,.08,0],[.61,0,.01,.02,0,.01,.03,.03,.01,.02,0,.09,.04,.05,.07]]],\n["L5H5","inducción: lo que vino después",[[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[.97,0,.01,.02,0,0,0,0,0,0,0,0,0,0,0],[.99,0,0,0,.01,0,0,0,0,0,0,0,0,0,0],[.97,0,.01,0,0,.01,0,0,0,0,0,0,0,0,0],[.98,0,.01,0,0,0,0,0,0,0,0,0,0,0,0],[.98,.01,.01,0,0,0,0,0,0,0,0,0,0,0,0],[.98,.01,0,0,0,0,0,0,0,0,0,0,0,0,0],[.94,.01,.01,0,.01,0,0,0,0,.02,0,0,0,0,0],[.07,0,0,0,0,.92,0,0,.01,0,0,0,0,0,0],[.94,0,0,0,0,.01,0,0,.01,0,.01,.02,0,0,0],[.97,.01,0,0,0,0,0,0,0,0,0,0,.01,0,0],[.97,.01,0,0,0,0,0,0,0,0,0,0,0,.01,0],[.9,.01,.01,0,0,0,0,.01,0,0,.01,0,.01,.02,.02]]],\n["L9H9","mueve un nombre",[[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[.94,0,.05,0,0,0,0,0,0,0,0,0,0,0,0],[.99,0,0,0,.01,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[.98,0,0,0,.01,0,.01,0,0,0,0,0,0,0,0],[.99,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[.99,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[.52,.01,.24,0,.21,0,0,0,0,0,0,0,0,0,0],[.96,0,0,0,.01,.01,0,0,0,0,0,0,0,0,0],[.35,.02,.51,0,.05,0,0,0,0,0,.05,0,0,0,0],[.98,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[.97,0,0,0,0,0,0,0,0,0,0,0,0,.01,0],[.18,.02,.67,0,.07,0,0,0,0,0,.06,0,0,0,0]]]];\ndocument.getElementById("sent").textContent=T.slice(1).join("");\nconst col=v=>`rgb(${26+v*81},${44+v*136},${35+v*121})`;\nconst views=H.map((h,k)=>{const w=document.createElement("div");w.className="h";w.style.left=k*428+"px";\n const g=document.createElement("div");g.className="g";const cs=[];for(let i=0;i<225;i++){const c=document.createElement("div");c.className="c";g.appendChild(c);cs.push(c)}\n w.appendChild(g);w.insertAdjacentHTML("beforeend",\'<div class="n"></div><div class="r"></div><div class="a"></div>\');\n w.children[1].textContent=h[0];w.children[2].textContent=h[1];document.body.appendChild(w);return {cs,a:w.children[3],m:h[2]}});\nlet q=1;\nfunction tick(){if(q==1)views.forEach(v=>v.cs.forEach(c=>c.style.background="#1A2C23"));\n document.getElementById("q").innerHTML=\'consulta: <b></b>\';document.querySelector("#q b").textContent=T[q];\n views.forEach(v=>{for(let j=0;j<15;j++)for(let i=0;i<=q;i++){const x=v.m[i][j];v.cs[i*15+j].style.background=i==q?(x>0.02?`rgb(${30+x*77},${60+x*152},${45+x*111})`:"#173A2A"):(j<=i?col(x):"#1A2C23")}\n  const r=v.m[q];let b=0;r.forEach((x,j)=>{if(x>r[b])b=j});v.a.innerHTML=\'→ <b></b> \'+Math.round(r[b]*100)+\'%\';v.a.querySelector("b").textContent=T[b]});\n q=q>=14?1:q+1}\ntick();setInterval(tick,1300);\n</script>')
ANIMS['block'] = (1664, 590, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\nsvg{position:absolute;left:0;top:0}\n.box{position:absolute;width:330px;height:84px;border-radius:16px;display:flex;align-items:center;justify-content:center;font-size:28px;font-weight:600;transition:all .4s}\n.vec{position:absolute;display:flex;gap:2px;transition:left .8s ease-in-out,top .8s ease-in-out,opacity .4s}\n.vec i{width:20px;height:30px;border-radius:3px;transition:background .6s}\n.plus{position:absolute;width:52px;height:52px;border-radius:26px;background:#0F1B16;color:#EAF2EA;font-size:36px;display:flex;align-items:center;justify-content:center}\n#eq{position:absolute;left:1000px;top:70px;width:660px}\n.e{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;padding:20px 24px;border-radius:14px;margin-bottom:18px;color:#7D8C82;transition:all .4s}\n.e.on{background:#0F1B16;color:#EAF2EA}\n#note{position:absolute;left:1000px;top:420px;width:640px;font-size:28px;line-height:1.4;color:#45554C}\n.lab{position:absolute;font-size:24px;color:#45554C}\n</style>\n<svg width="960" height="590"><line x1="240" y1="580" x2="240" y2="0" stroke="#3B82C4" stroke-width="14" stroke-linecap="round" opacity=".35"/>\n<path d="M240 430 H560 M720 388 V330 H266" fill="none" stroke="#13211A" stroke-width="4"/>\n<path d="M240 250 H560 M720 208 V150 H266" fill="none" stroke="#13211A" stroke-width="4"/></svg>\n<div class="lab" style="left:40px;top:540px">entra x</div><div class="lab" style="left:40px;top:10px">sale x′</div>\n<div class="lab" style="left:262px;top:500px;width:180px">residual stream</div>\n<div class="box" id="b1" style="left:560px;top:388px;background:#E1EADD">LayerNorm → Atención</div>\n<div class="box" id="b2" style="left:560px;top:208px;background:#E1EADD">LayerNorm → MLP</div>\n<div class="plus" style="left:214px;top:304px">+</div><div class="plus" style="left:214px;top:124px">+</div>\n<div id="eq"><div class="e" id="e1">x = x + Atención(LN(x))</div><div class="e" id="e2">x = x + MLP(LN(x))</div></div>\n<div id="note"><b>Atención</b> mueve información <i>entre</i> tokens. El <b>MLP</b> procesa cada token por separado. Ninguna sobrescribe el vector: solo le <b>suman</b> su contribución.</div>\n<script>\nconst N=8,c0=["#C9E3D2","#C9D9EA","#D3E8D9","#BFD3E8","#CDE6D5","#CFDDE9","#D6EADC","#C3D6EA"];\nfunction mk(){const v=document.createElement("div");v.className="vec";for(let i=0;i<N;i++)v.appendChild(document.createElement("i"));document.body.appendChild(v);return v}\nconst V=mk(),C=mk();const mv=(v,x,y)=>{v.style.left=x-N*11+"px";v.style.top=y-15+"px"};\nconst set=(v,cs)=>[...v.children].forEach((c,i)=>c.style.background=cs[i]);\nconst h1=["#2FA36B","#7FCDA3","#22865A","#A9DEC0","#4DB580","#93D5B1","#1E7A50","#BFE6CF"],h2=["#3B82C4","#7DB0DE","#2B6699","#A6C8E8","#5592CC","#8CBBE3","#3474B4","#B7D3EC"];\nconst mix=(a,b)=>a.map((x,i)=>{const p=s=>[1,3,5].map(k=>parseInt(s.slice(k,k+2),16));const A=p(x),B=p(b[i]);return `rgb(${A.map((v,j)=>Math.round(v*0.55+B[j]*0.45)).join(",")})`});\nlet cur;const e=k=>{["e1","e2"].forEach((id,i)=>document.getElementById(id).className="e"+(i+1==k?" on":""));["b1","b2"].forEach((id,i)=>{const b=document.getElementById(id);b.style.background=i+1==k?"#0F1B16":"#E1EADD";b.style.color=i+1==k?"#EAF2EA":"#13211A"})};\nconst at=(t,f)=>setTimeout(f,t);\nfunction run(){cur=c0.slice();set(V,cur);V.style.opacity=1;C.style.opacity=0;mv(V,240,560);mv(C,240,560);e(0);\n const sub=(t0,yb,yp,k,hc)=>{at(t0,()=>{mv(V,240,yb)});at(t0+800,()=>{set(C,cur);mv(C,240,yb);C.style.opacity=1;e(k)});\n  at(t0+1000,()=>mv(C,725,yb));at(t0+1900,()=>set(C,hc));at(t0+2600,()=>mv(C,725,yp));at(t0+3300,()=>{mv(C,240,yp);mv(V,240,yp)});\n  at(t0+4200,()=>{C.style.opacity=0;cur=mix(cur,hc);set(V,cur)})};\n sub(300,430,330,1,h1);sub(5000,250,150,2,h2);at(9800,()=>{mv(V,240,20);e(0)})}\nrun();setInterval(run,11500);\n</script>')
ANIMS['gpt'] = (1664, 400, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n.st{position:absolute;top:40px;height:250px;border-radius:20px;background:#E1EADD;border:3px solid #E1EADD;transition:all .35s;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px}\n.st.on{border-color:#2FA36B;background:#D3E8D9}\n.cap{position:absolute;top:310px;text-align:center;font-size:28px;font-weight:600}\n.sub{position:absolute;top:350px;text-align:center;font-size:24px;color:#45554C}\n.ar{position:absolute;top:150px;width:0;height:0;border-top:18px solid transparent;border-bottom:18px solid transparent;border-left:26px solid #AEBDB0}\n.tk{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:26px;background:#F2F5EF;border-radius:8px;padding:4px 12px;white-space:pre}\n.vec{display:flex;gap:3px}.vec i{width:16px;height:16px;border-radius:3px}\n.blk{position:absolute;width:300px;height:56px;border-radius:12px;background:#45554C;color:#F2F5EF;display:flex;align-items:center;justify-content:center;font-size:24px;font-weight:600;transition:background .3s}\n.blk.on{background:#2FA36B;color:#0F1B16}\n.br{display:flex;align-items:center;gap:10px;width:300px}\n.br span{width:110px;text-align:right;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:24px;white-space:pre}\n.br div{height:22px;border-radius:5px;background:#3B82C4;transition:width .6s}\n</style>\n<div class="st" id="S0" style="left:0;width:240px"><span class="tk">To</span><span class="tk"> be</span><span class="tk">…</span><span class="tk"> to</span></div>\n<div class="ar" style="left:254px"></div>\n<div class="st" id="S1" style="left:294px;width:240px"></div>\n<div class="ar" style="left:548px"></div>\n<div class="st" id="S2" style="left:588px;width:360px"></div>\n<div class="ar" style="left:962px"></div>\n<div class="st" id="S3" style="left:1002px;width:240px"><div style="font-size:26px;font-weight:600;text-align:center;line-height:1.3">LayerNorm<br>+ lineal<br>d → vocab</div></div>\n<div class="ar" style="left:1256px"></div>\n<div class="st" id="S4" style="left:1296px;width:368px;align-items:flex-start;padding-left:20px;box-sizing:border-box"></div>\n<div class="cap" style="left:0;width:240px">Tokens</div><div class="sub" style="left:0;width:240px">ids enteros</div>\n<div class="cap" style="left:294px;width:240px">Embeddings</div><div class="sub" style="left:294px;width:240px">token + posición</div>\n<div class="cap" style="left:588px;width:360px">N bloques transformer</div><div class="sub" style="left:588px;width:360px">atención + MLP, sumando</div>\n<div class="cap" style="left:1002px;width:240px">Unembedding</div><div class="sub" style="left:1002px;width:240px">vuelta al vocabulario</div>\n<div class="cap" style="left:1296px;width:368px">Logits</div><div class="sub" style="left:1296px;width:368px">siguiente token: « be»</div>\n<script>\nconst C=["#9ED6B5","#B9D5EF","#7FCDA3","#A9CBEA","#C8EAD5","#8CBBE3"];\nconst S1=document.getElementById("S1");for(let r=0;r<4;r++){const v=document.createElement("div");v.className="vec";for(let j=0;j<6;j++){const c=document.createElement("i");c.style.background=C[(j+r*2)%6];v.appendChild(c)}S1.appendChild(v)}\nconst S2=document.getElementById("S2"),B=[];for(let i=0;i<4;i++){const b=document.createElement("div");b.className="blk";b.style.left="30px";b.style.top=(180-i*52)+"px";b.textContent="bloque "+(i+1);S2.appendChild(b);B.push(b)}\nS2.style.position="absolute";\nconst P=[[" be",.984],[" have",.002],[" become",.002],[" exist",0]],S4=document.getElementById("S4"),bars=P.map(([t])=>{const r=document.createElement("div");r.className="br";r.innerHTML="<span></span><div></div>";r.children[0].textContent=t;S4.appendChild(r);return r.children[1]});\nconst full=()=>bars.forEach((b,i)=>b.style.width=Math.max(6,P[i][1]*170)+"px");full();\nconst st=i=>[0,1,2,3,4].forEach(k=>document.getElementById("S"+k).classList.toggle("on",k==i));\nconst at=(t,f)=>setTimeout(f,t);\nfunction run(){st(0);bars.forEach(b=>b.style.width="6px");B.forEach(b=>b.classList.remove("on"));\n at(900,()=>st(1));at(1800,()=>st(2));B.forEach((b,i)=>at(2000+i*500,()=>{B.forEach(x=>x.classList.remove("on"));b.classList.add("on")}));\n at(4200,()=>{B.forEach(x=>x.classList.remove("on"));st(3)});at(5100,()=>{st(4);full()});at(7000,()=>st(-1))}\nsetTimeout(()=>{run();setInterval(run,8500)},1500);\n</script>')
ANIMS['sampling'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\n#pr{position:absolute;left:0;top:0;font-size:30px;color:#45554C}\n#pr b{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;color:#13211A}\n#bars{position:absolute;left:0;top:70px;width:1000px}\n.r{display:flex;align-items:center;gap:20px;height:66px}\n.l{width:120px;text-align:right;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;white-space:pre}\n.b{height:42px;border-radius:8px;background:#3B82C4;transition:width .25s linear}\n.v{font-size:26px;color:#45554C;width:90px}\n#T{position:absolute;left:1100px;top:40px;width:560px;font-size:30px;color:#45554C}\n#tv{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:120px;font-weight:600;color:#13211A;line-height:1.1}\n#tag{font-size:30px;font-weight:600;color:#1B7347;margin-top:8px}\n#smp{position:absolute;left:0;top:440px;width:1664px;font-size:28px;color:#45554C}\n#smp span{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;background:#DAE4D5;border-radius:8px;padding:4px 10px;margin-right:8px;color:#13211A;white-space:pre;display:inline-block;margin-top:8px}\n</style>\n<div id="pr">Tras <b>“To be, or not to be,”</b> · top-5 de GPT-2, renormalizado</div>\n<div id="bars"></div>\n<div id="T">temperatura<div id="tv"></div><div id="tag"></div></div>\n<div id="smp"></div>\n<script>\nconst C=[[" a",0.236],[" an",0.049],[" is",0.034],[" the",0.022],[" or",0.02]];\nconst B=document.getElementById("bars"),rows=C.map(([t])=>{const r=document.createElement("div");r.className="r";r.innerHTML=\'<span class="l"></span><div class="b"></div><span class="v"></span>\';r.children[0].textContent=t;B.appendChild(r);return r});\nlet seed=3;const rnd=()=>{seed=(seed*16807)%2147483647;return seed/2147483647};\nfunction dist(T){const w=C.map(([,p])=>Math.pow(p,1/T)),z=w.reduce((a,b)=>a+b);return w.map(x=>x/z)}\nfunction draw(T){const d=dist(T);rows.forEach((r,i)=>{r.children[1].style.width=Math.max(4,d[i]*780)+"px";r.children[2].textContent=Math.round(d[i]*100)+"%"});\n document.getElementById("tv").textContent=T.toFixed(1);document.getElementById("tag").textContent=T<0.6?"casi determinista":T<1.3?"equilibrado":"creativo / caótico";\n const s=document.getElementById("smp");s.innerHTML="10 muestras: ";for(let k=0;k<10;k++){let u=rnd(),i=0;while(i<4&&u>d[i]){u-=d[i];i++}const e=document.createElement("span");e.textContent=C[i][0];s.appendChild(e)}}\nconst Ts=[0.2,0.4,0.6,0.8,1.0,1.3,1.6,2.0,1.6,1.3,1.0,0.6];let k=0;\ndraw(Ts[0]);setInterval(()=>{k=(k+1)%Ts.length;draw(Ts[k])},1300);\n</script>')
ANIMS['training'] = (1664, 620, '<style>\nhtml,body{margin:0;height:100%;background:#0F1B16;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#EAF2EA}\nsvg{position:absolute;left:0;top:0}\n.lab{position:absolute;font-size:24px;color:#A2B8AB}\n#out{position:absolute;left:960px;top:20px;width:700px}\n.k{font-size:24px;color:#A2B8AB;margin-bottom:10px}\npre{margin:0;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:24px;line-height:1.4;white-space:pre-wrap;background:#1A2C23;border-radius:14px;padding:20px 24px;min-height:40px}\n#before{color:#8A93A5;margin-bottom:26px}\n#after{color:#EAF2EA;min-height:300px}\n.lg{position:absolute;font-size:24px}\n</style>\n<svg id="s" width="900" height="620"></svg>\n<div class="lab" style="left:20px;top:0">pérdida (entropía cruzada)</div>\n<div class="lab" style="left:430px;top:580px">paso</div>\n<div class="lg" style="left:600px;top:200px;color:#6FAEE8">— entrenamiento</div><div class="lg" style="left:600px;top:240px;color:#6BD49C">— validación</div>\n<div id="out"><div class="k">paso 0 · pesos aleatorios</div><pre id="before">ROMEO:\nsuffering fliesglassere yourself captain feelsSword bells Autates begspacedumedatt</pre>\n<div class="k">paso 600</div><pre id="after"></pre></div>\n<script>\nconst st=[0,100,200,300,400,500,600],tr=[9.478,5.766,5.073,4.847,4.568,4.419,4.237],va=[9.472,5.912,5.469,5.155,4.930,4.868,4.837];\nconst NS="http://www.w3.org/2000/svg",s=document.getElementById("s"),X=v=>90+v/600*740,Y=v=>520-(v-4)/6*460;\nfunction el(n,a){const e=document.createElementNS(NS,n);for(const k in a)e.setAttribute(k,a[k]);s.appendChild(e);return e}\nel("line",{x1:90,y1:520,x2:840,y2:520,stroke:"#3A5046","stroke-width":2});el("line",{x1:90,y1:60,x2:90,y2:520,stroke:"#3A5046","stroke-width":2});\n[4,6,8,10].forEach(v=>{el("line",{x1:90,y1:Y(v),x2:840,y2:Y(v),stroke:"#20332A","stroke-width":1});const t=el("text",{x:50,y:Y(v)+8,fill:"#A2B8AB","font-size":24,"text-anchor":"middle"});t.textContent=v});\n[0,200,400,600].forEach(v=>{const t=el("text",{x:X(v),y:556,fill:"#A2B8AB","font-size":24,"text-anchor":"middle"});t.textContent=v});\nconst pv=el("line",{x1:90,y1:Y(Math.log(11706)),x2:840,y2:Y(Math.log(11706)),stroke:"#56695E","stroke-dasharray":"8 8","stroke-width":2});\nconst tl=el("text",{x:840,y:Y(Math.log(11706))-12,fill:"#A2B8AB","font-size":24,"text-anchor":"end"});tl.textContent="azar: ln(11.706) ≈ 9,4";\nconst mk=(d,c)=>el("path",{d:"M"+st.map((x,i)=>X(x)+" "+Y(d[i])).join(" L"),fill:"none",stroke:c,"stroke-width":6,"stroke-linecap":"round","stroke-linejoin":"round"});\nconst p1=mk(tr,"#6FAEE8"),p2=mk(va,"#6BD49C");\nconst A="ROMEO:\\nThen, the duke;\\nThan of mine own king, and my father,\\nIn a husband, of her hand.\\n\\nBUCKINGHAM:\\nI pray you have done, my love!";\nfunction run(){[p1,p2].forEach(p=>{const L=p.getTotalLength();p.style.transition="none";p.style.strokeDasharray=L;p.style.strokeDashoffset=L;p.getBoundingClientRect();p.style.transition="stroke-dashoffset 4s ease-out";p.style.strokeDashoffset=0});\n const o=document.getElementById("after");o.textContent="";let i=0;setTimeout(()=>{const iv=setInterval(()=>{o.textContent=A.slice(0,++i);if(i>=A.length)clearInterval(iv)},35)},4200)}\nrun();setInterval(run,14000);\n</script>')
print("Estilo del taller listo ·", "Google Sans" if _sans else "fuente por defecto", "·", len(ANIMS), "animaciones")

## 1. Texto → tokens

Un modelo de lenguaje no ve letras ni palabras: ve **enteros**. El tokenizador corta el texto en trozos (*tokens*) y asigna a cada uno un identificador.

Usamos el corpus *Tiny Shakespeare* (~1 MB de texto). Pero antes, recordemos qué hace un LLM: dar una probabilidad a cada posible token siguiente (probabilidades reales de GPT-2).

In [ ]:
anim("nexttoken")

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")
print(f"{len(text):,} caracteres\n")
print(text[:400])

### Tokenización por caracteres vs. BPE

La opción más simple es un token por carácter. GPT-2 (y casi todos los LLM) usan **Byte Pair Encoding (BPE)**: empiezan con bytes y van fusionando los pares más frecuentes hasta tener un vocabulario de ~50k trozos. Las palabras frecuentes quedan como un solo token; las raras se parten en sub-palabras, así que **nunca hay palabras desconocidas**.

Así se construye el vocabulario, fusionando una y otra vez el par más frecuente:

In [ ]:
anim("bpe")

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

def show_tokens(s):
    ids = tokenizer.encode(s)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{len(s)} caracteres → {len(ids)} tokens")
    print(" | ".join(repr(p) for p in pieces))
    print(ids, "\n")

show_tokens("To be, or not to be: that is the question.")
show_tokens("Hola, bienvenidos al taller de interpretabilidad mecanicista.")  # 🧪 prueba otras frases
show_tokens("Ourense 2026 supercalifragilisticoespialidoso")

In [ ]:
anim("tokenize")

**Observa:** el espacio va pegado a la palabra siguiente (`' be'`), el castellano se trocea mucho más que el inglés (el vocabulario de GPT-2 se aprendió sobre texto en inglés) y una palabra inventada se descompone sin problema.

### Vocabulario compacto

GPT-2 tiene 50 257 tokens, pero Shakespeare solo usa ~12k. Para que nuestro modelo sea pequeño y rápido, re-indexamos solo los tokens que aparecen. (Es un truco práctico; el modelo sigue usando la tokenización BPE de GPT-2.)

In [ ]:
gpt2_ids = tokenizer.encode(text)
vocab = sorted(set(gpt2_ids))
gpt2_to_ours = {g: i for i, g in enumerate(vocab)}
ours_to_gpt2 = torch.tensor(vocab)

def encode(s):  return [gpt2_to_ours[g] for g in tokenizer.encode(s)]
def decode(ids): return tokenizer.decode(ours_to_gpt2[torch.as_tensor(ids)].tolist())

data = torch.tensor([gpt2_to_ours[g] for g in gpt2_ids])
VOCAB_SIZE = len(vocab)
print(f"{len(data):,} tokens, vocabulario de {VOCAB_SIZE:,}")

n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

### Pares entrada → objetivo

Entrenar un LLM = **predecir el siguiente token**. Cogemos una ventana de `context_length` tokens; el objetivo es la misma ventana desplazada una posición. Cada ventana contiene en realidad `context_length` ejemplos de entrenamiento a la vez.

In [ ]:
anim("window")

In [ ]:
context_length = 8
x = data[:context_length]
y = data[1:context_length + 1]
for t in range(context_length):
    print(f"{decode(x[:t+1])!r:45} → {decode(y[t:t+1])!r}")

In [ ]:
def get_batch(split, batch_size, context_length):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - context_length - 1, (batch_size,))
    x = torch.stack([src[i:i + context_length] for i in ix])
    y = torch.stack([src[i + 1:i + context_length + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", batch_size=4, context_length=8)
print("entradas:", xb.shape, "\n", xb)
print("objetivos:", yb.shape)

## 2. Tokens → vectores: embeddings

### ¿Para qué queremos un embedding?

Un id es solo una etiqueta. Con *one-hot* (un vector de ceros con un 1) todas las palabras quedan **a la misma distancia** entre sí y los vectores son enormes. Un embedding es un vector denso y pequeño que se **aprende**: al entrenar, lo que se usa en contextos parecidos acaba cerca.

In [ ]:
anim("whyembed")

In [ ]:
# One-hot: el producto escalar entre dos palabras distintas es SIEMPRE 0
palabras = [" king", " queen", " London"]
ids = [encode(w)[0] for w in palabras]
onehot = F.one_hot(torch.tensor(ids), VOCAB_SIZE).float()
print("tamaño de cada vector one-hot:", onehot.shape[1])
print("similitudes one-hot:\n", (onehot @ onehot.T).numpy())
# Al final del notebook veremos qué parecidos aprende un embedding entrenado.

### Embedding de token + embedding de posición

Cada ID se convierte en un vector aprendido con `nn.Embedding`, que no es más que una tabla de búsqueda `[vocab, d_model]`.

La atención, por sí sola, no sabe el **orden** de los tokens. Por eso se suma un segundo embedding, de **posición** (GPT-2 usa posiciones absolutas aprendidas).

In [ ]:
anim("embed")

In [ ]:
d_model = 16
tok_emb = nn.Embedding(VOCAB_SIZE, d_model)
pos_emb = nn.Embedding(context_length, d_model)

xb, _ = get_batch("train", 4, context_length)
t = tok_emb(xb.cpu())                          # [batch, seq, d_model]
p = pos_emb(torch.arange(context_length))      # [seq, d_model] (se difunde sobre el batch)
h = t + p
print("token emb:", t.shape, "| pos emb:", p.shape, "| entrada al transformer:", h.shape)

A partir de aquí, cada token es un vector de `d_model` números. Ese vector viajará por todo el modelo y se irá modificando capa a capa: lo llamaremos el **residual stream** en la parte 2.

## 3. Atención

La idea: cada token construye un nuevo vector como **media ponderada** de los vectores de los demás tokens. Los pesos dicen "cuánto me importa cada otro token".

### 3.1 Versión mínima (sin pesos entrenables)

Frase de juguete con embeddings de 3 dimensiones inventados a mano:

In [ ]:
words = ["Tu", "viaje", "empieza", "con", "un", "paso"]
inputs = torch.tensor([
    [0.43, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.64],
    [0.22, 0.58, 0.33],
    [0.77, 0.25, 0.10],
    [0.05, 0.80, 0.55],
])

scores = inputs @ inputs.T              # similitud = producto escalar entre todos los pares
weights = torch.softmax(scores, dim=-1) # cada fila suma 1
context = weights @ inputs              # media ponderada
print("pesos de atención de 'viaje':", weights[1].numpy().round(2))
print("vector de contexto de 'viaje':", context[1].numpy().round(3))

In [ ]:
def plot_attn(w, labels, title=""):
    w = w.detach().cpu(); top = max(0.3, w.max().item())
    fig, ax = plt.subplots(figsize=(5, 4.4))
    ax.imshow(w, cmap="taller", vmin=0, vmax=top)
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            ax.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color=PAPER if w[i, j] > 0.6 * top else INK)
    ax.set_xticks(range(len(labels)), labels, rotation=90); ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("clave (a quién miro)"); ax.set_ylabel("consulta (quién mira)")
    ax.tick_params(length=0); [sp.set_visible(False) for sp in ax.spines.values()]
    ax.set_title(title); plt.show()

plot_attn(weights, words, "Atención simple")

In [ ]:
anim("simpleattn")

### 3.2 ¿No debería cada token atenderse solo a sí mismo?

Si la atención mide parecido, y lo más parecido a mí soy yo, la diagonal debería valer ~1. **No pasa**: la softmax reparte entre puntuaciones parecidas y, sobre todo, el producto escalar **premia los vectores grandes**. Además la puntuación es simétrica y no tiene nada que aprender. Por eso necesitamos Q, K y V.

In [ ]:
anim("selfattn")

In [ ]:
diag = weights.diagonal()
norms = inputs.norm(dim=1)
for i, w in enumerate(words):
    top = weights[i].argmax().item()
    print(f"{w:8} se atiende a sí mismo {diag[i]:.2f} · al que más mira: {words[top]:8} ({weights[i, top]:.2f}) · norma {norms[i]:.2f}")

### 3.3 Query, Key, Value

En el transformer real, cada token proyecta su vector en tres papeles mediante matrices entrenables:

- **Query** $q = xW_Q$: "¿qué estoy buscando?"
- **Key** $k = xW_K$: "¿qué ofrezco para ser encontrado?"
- **Value** $v = xW_V$: "¿qué información aporto si me eligen?"

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Dividir por $\sqrt{d_k}$ evita que los productos escalares crezcan con la dimensión y saturen la softmax.

In [ ]:
torch.manual_seed(0)
d_in, d_out = 3, 2
W_q, W_k, W_v = (nn.Linear(d_in, d_out, bias=False) for _ in range(3))
Q, K, V = W_q(inputs), W_k(inputs), W_v(inputs)
scores = Q @ K.T / math.sqrt(d_out)
weights = torch.softmax(scores, dim=-1)
print(weights.detach().numpy().round(2))

Con matrices aleatorias los pesos salen casi uniformes: **todavía no hay nada que buscar**. Es el entrenamiento el que da forma a $W_Q$ y $W_K$ para que ciertas consultas encajen con ciertas claves.

### 3.4 Máscara causal

Para predecir el siguiente token, un token **no puede mirar al futuro**. Se pone $-\infty$ en las posiciones por encima de la diagonal antes de la softmax, y así reciben peso 0.

In [ ]:
T = len(words)
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
causal_weights = torch.softmax(scores.masked_fill(mask, float("-inf")), dim=-1)
plot_attn(causal_weights, words, "Atención causal")

In [ ]:
anim("mask")

### 3.5 Multi-head attention

En lugar de una sola atención, usamos varias **cabezas** en paralelo, cada una con sus propias $W_Q, W_K, W_V$ de dimensión `d_model / n_heads`. Cada cabeza puede especializarse en una relación distinta (token anterior, sujeto–verbo, etc. — lo veremos en la parte 2). Al final se concatenan y se mezclan con una proyección de salida $W_O$.

Así se ven 4 cabezas reales de GPT-2 recorriendo una frase:

In [ ]:
anim("multihead")

Guardamos los pesos de atención en `self.last_attn` para poder visualizarlos luego.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_length, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), 1))
        self.last_attn = None

    def forward(self, x):
        B, T, D = x.shape
        # [B, T, D] -> [B, heads, T, d_head]
        split = lambda t: t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = split(self.W_q(x)), split(self.W_k(x)), split(self.W_v(x))

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)       # [B, H, T, T]
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        self.last_attn = attn.detach()
        out = self.dropout(attn) @ v                                     # [B, H, T, d_head]
        out = out.transpose(1, 2).reshape(B, T, D)                       # concatenar cabezas
        return self.W_o(out)

mha = MultiHeadAttention(d_model=16, n_heads=4, context_length=8)
print(mha(h).shape)   # misma forma que la entrada: [batch, seq, d_model]

## 4. El GPT completo

Un **bloque transformer** (estilo GPT-2, *pre-LayerNorm*):

```
x = x + Atención(LayerNorm(x))
x = x + MLP(LayerNorm(x))
```

- **LayerNorm**: normaliza cada vector (media 0, varianza 1) y aplica una escala y desplazamiento aprendidos. Estabiliza el entrenamiento.
- **MLP / feed-forward**: dos capas lineales (expande ×4, GELU, contrae). Opera sobre cada token por separado; es donde se cree que se almacena mucho "conocimiento".
- **Conexiones residuales** (`x = x + ...`): cada subcapa *suma* su contribución a un mismo vector. Esto es clave para la interpretabilidad: el residual stream es un canal de comunicación compartido que todas las capas leen y escriben.

In [ ]:
anim("block")

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps, self.scale, self.shift = eps, nn.Parameter(torch.ones(d)), nn.Parameter(torch.zeros(d))
    def forward(self, x):
        mean, var = x.mean(-1, keepdim=True), x.var(-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):  # aproximación usada en GPT-2
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

xs = torch.linspace(-4, 4, 200)
plt.figure(figsize=(5.5, 3)); plt.plot(xs, GELU()(xs), label="GELU"); plt.plot(xs, F.relu(xs), "--", label="ReLU")
plt.axhline(0, color=LINE, lw=1); plt.legend(); plt.title("GELU vs. ReLU"); plt.show()

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, 4 * d_model), GELU(),
                                 nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = LayerNorm(cfg["d_model"]), LayerNorm(cfg["d_model"])
        self.attn = MultiHeadAttention(cfg["d_model"], cfg["n_heads"], cfg["context_length"], cfg["dropout"])
        self.mlp = FeedForward(cfg["d_model"], cfg["dropout"])
        self.drop = nn.Dropout(cfg["dropout"])
    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))   # escribe en el residual stream
        x = x + self.drop(self.mlp(self.ln2(x)))    # escribe en el residual stream
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.ln_f = LayerNorm(cfg["d_model"])
        self.unembed = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)  # vector → logits

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.blocks(self.drop(x))
        return self.unembed(self.ln_f(x))           # [B, T, vocab]

### El diagrama completo, de punta a punta

In [ ]:
anim("gpt")

### Configuración

Como referencia, GPT-2 small: `d_model=768, n_layers=12, n_heads=12, context_length=1024`, 124M parámetros. El nuestro es ~100 veces más pequeño para que entrene en minutos. 🧪 Puedes cambiar la configuración, pero ten en cuenta el tiempo de entrenamiento.

In [ ]:
CFG = {
    "vocab_size": VOCAB_SIZE,
    "context_length": 64,
    "d_model": 128,
    "n_heads": 4,
    "n_layers": 4,
    "dropout": 0.1,
}
torch.manual_seed(123)
model = GPTModel(CFG).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.2f} M parámetros")
for name, p in list(model.named_parameters())[:12]:
    print(f"  {name:32} {tuple(p.shape)}")

### Generar texto

La salida son **logits**: una puntuación por cada token del vocabulario en cada posición. Para generar, nos quedamos con la última posición, la convertimos en probabilidades, elegimos un token, lo añadimos y repetimos.

- `temperature` < 1 → más conservador; > 1 → más creativo/caótico.
- `top_k`: solo muestrea entre los k tokens más probables.

In [ ]:
anim("sampling")

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=60, temperature=1.0, top_k=None):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(max_new_tokens):
        logits = model(idx[:, -CFG["context_length"]:])[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")
        next_id = torch.multinomial(torch.softmax(logits, -1), 1)
        idx = torch.cat([idx, next_id], dim=1)
    return decode(idx[0].tolist())

print(generate(model, "ROMEO:\n", 40))

Sin entrenar, el modelo produce tokens al azar. **Este es el "foundation model" listo para entrenar**: toda la arquitectura está, solo faltan los pesos.

## 5. Entrenamiento

La pérdida es la **entropía cruzada** entre los logits y el token real siguiente. Con pesos aleatorios debería rondar $\ln(\text{vocab}) \approx 9.4$.

El bucle es el estándar de PyTorch: batch → forward → pérdida → backward → paso del optimizador. Esto es lo que obtuvimos en un run de prueba en CPU:

In [ ]:
anim("training")

In [ ]:
def compute_loss(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits.flatten(0, 1), y.flatten())

@torch.no_grad()
def estimate_loss(model, iters=20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = [compute_loss(model, *get_batch(split, 32, CFG["context_length"])).item() for _ in range(iters)]
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

print(estimate_loss(model), " | ln(vocab) =", round(math.log(VOCAB_SIZE), 2))

In [ ]:
# 🧪 Más pasos = mejor modelo. En una GPU T4, 2000 pasos tardan ~1-2 min;
#    en CPU, 600 pasos tardan ~5 min (depende de los núcleos).
MAX_STEPS = 2000 if device == "cuda" else 600
BATCH_SIZE = 32
EVAL_EVERY = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
history = {"step": [], "train": [], "val": []}
t0 = time.time()
model.train()
for step in range(MAX_STEPS + 1):
    if step % EVAL_EVERY == 0:
        l = estimate_loss(model)
        history["step"].append(step); history["train"].append(l["train"]); history["val"].append(l["val"])
        print(f"paso {step:5d} | train {l['train']:.3f} | val {l['val']:.3f} | {time.time()-t0:5.0f}s")
    if step == MAX_STEPS: break
    x, y = get_batch("train", BATCH_SIZE, CFG["context_length"])
    loss = compute_loss(model, x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.axhline(math.log(VOCAB_SIZE), color="#7D8C82", ls="--", lw=1.2)
plt.text(history["step"][-1], math.log(VOCAB_SIZE) + 0.15, f"azar: ln({VOCAB_SIZE:,}) ≈ {math.log(VOCAB_SIZE):.1f}".translate(str.maketrans(",.", ".,")), ha="right", color=MUTED, fontsize=9)
plt.plot(history["step"], history["train"], color=BLUE, label="entrenamiento")
plt.plot(history["step"], history["val"], color=GREEN, label="validación")
plt.xlabel("paso"); plt.ylabel("entropía cruzada"); plt.grid(axis="y"); plt.legend(); plt.title("Curva de pérdida"); plt.show()

In [ ]:
# 🧪 Juega con el prompt, la temperatura y top_k
print(generate(model, "ROMEO:\n", max_new_tokens=120, temperature=0.8, top_k=40))

Con unos minutos de entrenamiento el modelo ya aprende el **formato** (nombres en mayúsculas, dos puntos, versos cortos) y bastante vocabulario, aunque el significado sea flojo. Si la pérdida de validación empieza a subir mientras la de entrenamiento baja, estamos **sobreajustando**: el corpus es muy pequeño.

### ¿Qué ha aprendido el embedding?

Volvamos a la pregunta del principio. Buscamos los vecinos más cercanos (similitud coseno) de algunas palabras en la tabla de embeddings.

In [ ]:
E = F.normalize(model.tok_emb.weight.detach(), dim=1)
def neighbours(word, k=6):
    i = encode(word)[0]
    sims = E @ E[i]
    best = sims.topk(k + 1).indices[1:]
    return [(decode([j.item()]), round(sims[j].item(), 2)) for j in best]

for w in [" king", " love", " father", " Rome"]:   # 🧪 prueba otras (con espacio delante)
    print(f"{w!r:10} →", neighbours(w))

**Spoiler:** con 3,8 M de parámetros y unos minutos de Shakespeare, los vecinos son todavía casi aleatorios. El modelo ya ha aprendido el *formato* del texto, pero una geometría útil de las palabras necesita muchísimo más texto. En la parte 2 repetimos esta misma celda con GPT-2, entrenado con ~40 GB de texto, y ahí sí aparecen los parecidos.

Guardamos los pesos por si queréis seguir jugando después:

In [ ]:
torch.save({"cfg": CFG, "state_dict": model.state_dict(), "vocab": vocab}, "mini_gpt_shakespeare.pt")

## 6. Primer vistazo dentro del modelo

Cada `MultiHeadAttention` guardó sus pesos de atención en la última pasada. Miremos qué hace cada cabeza en una frase.

Es habitual ver cabezas que miran casi siempre al **token anterior**, otras que miran al **primer token** (un "sumidero" donde aparcar atención cuando no hay nada útil) y otras con patrones más difusos. En la parte 2 haremos esto con rigor sobre GPT-2.

In [ ]:
@torch.no_grad()
def show_heads(model, prompt, layer):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    model(idx)
    attn = model.blocks[layer].attn.last_attn[0].cpu()        # [heads, T, T]
    labels = [decode([i]) for i in idx[0].tolist()]
    H = attn.shape[0]
    fig, axes = plt.subplots(1, H, figsize=(3.3 * H, 3.5))
    for hh in range(H):
        axes[hh].imshow(attn[hh], cmap="taller", vmin=0, vmax=1); axes[hh].tick_params(length=0)
        axes[hh].set_xticks(range(len(labels))); axes[hh].set_xticklabels(labels, rotation=90, fontsize=7)
        axes[hh].set_yticks(range(len(labels))); axes[hh].set_yticklabels(labels, fontsize=7)
        axes[hh].set_title(f"capa {layer} · cabeza {hh}")
    plt.tight_layout(); plt.show()
    return attn

prompt = "KING HENRY:\nWhat says my lord? speak, I pray thee, speak."
for layer in range(CFG["n_layers"]):   # 🧪 cambia el prompt
    _ = show_heads(model, prompt, layer)

### Una medida simple: ¿qué cabezas miran al token anterior?

Media del peso en la subdiagonal (posición $i$ → $i-1$). Un valor cercano a 1 indica una **cabeza de token anterior**.

In [ ]:
@torch.no_grad()
def prev_token_scores(model, n_samples=8):
    model.eval()
    x, _ = get_batch("val", n_samples, CFG["context_length"])
    model(x)
    scores = torch.zeros(CFG["n_layers"], CFG["n_heads"])
    for l, block in enumerate(model.blocks):
        a = block.attn.last_attn                                       # [B, H, T, T]
        scores[l] = a.diagonal(offset=-1, dim1=-2, dim2=-1).mean(dim=(0, -1)).cpu()
    return scores

s = prev_token_scores(model)
plt.figure(figsize=(4, 3.5))
plt.imshow(s, cmap="taller", vmin=0, vmax=1)
plt.xticks(range(CFG["n_heads"])); plt.yticks(range(CFG["n_layers"]))
plt.xlabel("cabeza"); plt.ylabel("capa"); plt.title("Puntuación de token anterior")
for l in range(s.shape[0]):
    for hh in range(s.shape[1]):
        plt.text(hh, l, f"{s[l, hh]:.2f}", ha="center", va="center", fontsize=9, color=PAPER if s[l, hh] > 0.5 else INK)
plt.show()

## 🧪 Retos (si te sobra tiempo)

1. **Sin posiciones**: quita `pos_emb` del `forward` y re-entrena. ¿Qué le pasa a la generación? ¿Por qué?
2. **Sin máscara causal**: comenta la línea del `masked_fill`. La pérdida baja muchísimo… ¿por qué es una trampa?
3. **Una sola cabeza** (`n_heads=1`) vs. 8 cabezas: compara pérdida final.
4. **Temperatura**: genera con `temperature=0.1` y `temperature=2.0`. Describe la diferencia.
5. Busca en las figuras de la sección 6 una cabeza que no sea de "token anterior" e intenta describir qué patrón sigue.

---
**Siguiente:** en la parte 2 abrimos GPT-2 small (124M parámetros, entrenado de verdad) con *TransformerLens* y buscamos circuitos concretos.